# Task 03: LangChain Support Agent

Build a full customer support agent using `create_agent` from LangChain 1.x. The agent must use multiple tools to autonomously handle support queries.

**API**: `create_agent(llm, tools, system_prompt=...)` → `.invoke({"messages": [...]})`

## Setup

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

OPENAI_API_KEY = "your-api-key-here"
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
print("✓ LLM initialized")


In [ ]:
import json, os

# In Jupyter, os.getcwd() returns the notebook's directory (learning/, tasks/, solutions/)
# Fixtures are one level up at ../fixtures/input/
fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))

with open(os.path.join(fixtures, "tickets.json")) as f:
    tickets = json.load(f)

with open(os.path.join(fixtures, "knowledge_base.json")) as f:
    knowledge_base = json.load(f)

with open(os.path.join(fixtures, "test_questions.json")) as f:
    test_questions = json.load(f)

print(f"✓ Loaded {len(tickets)} tickets, {len(knowledge_base)} KB articles, {len(test_questions)} test questions")


## Part 1: Define the Tools

You need three tools:

1. **`classify_ticket`** — calls an LCEL chain to classify category + priority
2. **`search_knowledge_base`** — RAG over `knowledge_base.json` using FAISS
3. **`escalate_to_human`** — logs escalation (no LLM call needed)

Remember: the **docstring is the agent's guide** — write specific, actionable descriptions.

In [ ]:
# YOUR CODE HERE
# Implement all three tools using @tool decorator



In [ ]:
# TEST — Tool structure checks (no API call)
for tool_fn in [classify_ticket, search_knowledge_base, escalate_to_human]:
    assert hasattr(tool_fn, 'name'), f"{tool_fn} must be a LangChain tool"
    assert hasattr(tool_fn, 'description'), f"{tool_fn} must have a description"
    assert len(tool_fn.description) > 50, f"{tool_fn.name} docstring too short — be specific"
    print(f"  ✓ {tool_fn.name!r}: {tool_fn.description[:60]}...")

print("\n✓ All three tools defined correctly")


In [ ]:
# TEST — Tools callable with invoke
import json

result = classify_ticket.invoke({"ticket_text": "I was charged twice this billing cycle"})
assert isinstance(result, str), "classify_ticket must return a string"
parsed = json.loads(result)
assert "category" in parsed and "priority" in parsed, "classify_ticket must return JSON with category and priority"
print(f"✓ classify_ticket works: {parsed}")

kb_result = search_knowledge_base.invoke({"query": "refund policy"})
assert isinstance(kb_result, str) and len(kb_result) > 50, "search_knowledge_base must return non-empty text"
print(f"✓ search_knowledge_base works: {kb_result[:80]}...")

esc_result = escalate_to_human.invoke({"reason": "account compromise", "priority": "high"})
assert isinstance(esc_result, str)
print(f"✓ escalate_to_human works: {esc_result}")


## Part 2: Build the Agent

Create an `agent` using `create_agent` with:
- A system prompt instructing: classify → search → respond → escalate if urgent
- All three tools

Invoke with `{"messages": [{"role": "user", "content": "..."}]}`

In [ ]:
# YOUR CODE HERE
# from langchain.agents import create_agent
# Define: agent



In [ ]:
# TEST — Agent has .invoke() method
assert hasattr(agent, 'invoke'), "agent must have .invoke() method"

# Quick structural test: invoke must return dict with 'messages'
test_result = agent.invoke({
    "messages": [{"role": "user", "content": "Hello"}]
})
assert "messages" in test_result, "agent.invoke() must return dict with 'messages' key"
assert len(test_result["messages"]) > 0, "messages list must not be empty"
print("✓ Agent structure correct")
print(f"  Final message type: {type(test_result['messages'][-1]).__name__}")


## Part 3: Run the Agent on Real Queries

In [ ]:
def ask_agent(agent, question):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content

# Test 1: Policy question — agent should search KB
print("=" * 60)
print("Test 1: API rate limit question")
print("=" * 60)
answer1 = ask_agent(agent, "Why is my API returning 429 errors? How do I fix this?")
print("Answer:", answer1)


In [ ]:
# TEST — Policy answer quality
assert len(answer1) > 50, "Answer too short"
found_kw = any(kw in answer1.lower() for kw in ["429", "rate limit", "retry", "too many"])
assert found_kw, "Answer should mention rate limiting — did the agent search the KB?"
print("✓ API rate limit answer is informative")


In [ ]:
# Test 2: Urgent security issue — agent should escalate or provide security steps
print("=" * 60)
print("Test 2: Compromised account (high priority)")
print("=" * 60)
answer2 = ask_agent(agent, "My account was hacked! I see login attempts from Russia. I need this fixed IMMEDIATELY.")
print("Answer:", answer2)


In [ ]:
# TEST — Urgent issue handling
assert len(answer2) > 30, "Answer too short for urgent issue"
handled = any(kw in answer2.lower() for kw in ["high", "escalat", "lock", "password", "2fa", "security", "immediately"])
assert handled, "Agent should handle urgent security issue — check tools and system prompt"
print("✓ Urgent issue handled appropriately")


In [ ]:
# Test 3: Cancellation policy
print("=" * 60)
print("Test 3: Cancellation and data retention")
print("=" * 60)
answer3 = ask_agent(agent, "What happens to my data if I cancel my subscription?")
print("Answer:", answer3)


In [ ]:
# TEST — Cancellation answer quality
found_kw = any(kw in answer3.lower() for kw in ["90 days", "90", "export", "delet", "cancel"])
assert found_kw, "Answer should mention data retention period — did the agent find the cancellation article?"
print("✓ Cancellation policy answer is informative")


## Part 4: Inspect Tool Calls

Filter messages for tool calls to verify the agent actually used tools.

In [ ]:
# YOUR CODE HERE
# Run the agent on a question, then inspect which tools were called
# Hint: filter result["messages"] for ToolMessage instances



In [ ]:
# TEST — Agent uses tools
from langchain_core.messages import ToolMessage

assert 'tool_messages' in dir() or 'tool_messages' in locals(),     "Define tool_messages by filtering result['messages'] for ToolMessage instances"

assert len(tool_messages) > 0,     "Agent made no tool calls — check your system prompt and tool descriptions"

tool_names_used = [m.name for m in tool_messages]
print(f"✓ Agent made {len(tool_messages)} tool call(s): {tool_names_used}")
assert "search_knowledge_base" in tool_names_used or "classify_ticket" in tool_names_used,     "Agent should use at least search_knowledge_base or classify_ticket"
print("✓ Agent used tools correctly")
